# MRP Engine — tutorial and shared runner for any domain

This notebook runs one domain's MRP through the generic MRP engine. It is a **tutorial**: it contains no model logic, and every step
calls the installed package (`pip install cowmrp[plot]`). The steps are the same Step 0-10 sequence as before; steps 1-4 are shown one
stage at a time so you can see what a domain provides, and `cowmrp.model.build_model` then does all four in one call.

The domain is chosen by the environment variable `MRP_DOMAIN` (default `cow`; also `youngstock`, or the dotted path of your own package).
`COW_TIME_UNIT` (`month`, `week`, `day`) sets the time resolution.

**How to use**
- From a module runner: `python runners/<module>/run.py --notebook`, or `python runners/_shared/run_notebook.py <domain>`.
  The executed copy and the exports go to `runners/<domain>/outputs/notebook/`.
- Interactively: set `MRP_DOMAIN` before starting Jupyter (or edit `DOMAIN` in the first cell) and run all cells top to bottom.

**What a domain provides** (see `cowmrp.domain`): `states.py` (`build_<domain>_states`), `probabilities`, `transitions.py`, optionally `rewards`,
and `parameters.py` (`TIME_UNIT`, `UNIT_DAYS`). Optional extras live in `notebook_hooks.py`; a step whose hook is missing is skipped.

```
cowmrp.mrp      <- generic MRP mathematics (knows no animal)
cowmrp.domain   <- the interface a domain provides         cowmrp.model  <- build / solve any domain
cowmrp.domains.<domain>   <- the domain's content          this notebook <- just a runner
```

In [ ]:
import os
import numpy as np
import pandas as pd

from cowmrp.domain import load_domain, validate_domain
from cowmrp.mrp import core                      # the generic engine
from cowmrp.mrp.utils import matrix_checksum, reachable_states, row_sum_report
from cowmrp.model import (build_model, evaluate, per_step_gamma, initial_distribution, evolve_distribution,
                          replacement_split, simulate, monte_carlo, group_summary)
from cowmrp.io import export_model

# DOMAIN: from the environment (set by runners/_shared/run_notebook.py), 'cow' when run by hand.
DOMAIN = os.environ.get("MRP_DOMAIN", "cow")
OUT_DIR = os.environ.get("MRP_OUT_DIR", "mrp_tutorial_output")     # where the exports of Step 8 go
os.makedirs(OUT_DIR, exist_ok=True)

dom = load_domain(DOMAIN)
print("cowmrp tutorial: package imported successfully.")
print(f"Domain: {DOMAIN}   outputs: {os.path.abspath(OUT_DIR)}")

## Step 0: Set Parameters
`DOMAIN` is read in the first cell. Below: the evaluation settings, then the domain's own parameters and optional hooks.

In [ ]:
# USER SETTINGS (MRP evaluation parameters)
GAMMA_ANNUAL = 0.95      # 5% annual discount rate
NUM_YEARS = 5            # How many years to evaluate

TIME_UNIT, UNIT_DAYS = dom.time_unit, dom.unit_days
STRICT = dom.strict                                          # False: rows that do not sum to 1 are reported, not fatal
HAS_REWARDS = dom.has_rewards
GROUP_NAME = dom.hook("GROUP_NAME", "Group")
print(f"Domain: {DOMAIN.upper()}   rewards: {'yes' if HAS_REWARDS else 'no'}   interface problems: {validate_domain(dom) or 'none'}")

gamma = per_step_gamma(dom, GAMMA_ANNUAL)
print("=" * 60)
print("TIME RESOLUTION (single source of truth: the domain's own parameters module)")
print("=" * 60)
print(f"TIME_UNIT  = {TIME_UNIT!r}   (1 MRP step = {UNIT_DAYS} real day(s))")
if os.environ.get("COW_TIME_UNIT") is not None:
    print(f"NOTE: COW_TIME_UNIT is set to {os.environ['COW_TIME_UNIT']!r} in this process.")
print(f"Annual discount {GAMMA_ANNUAL} -> per-step gamma {gamma:.6f}   ({core.steps_per_year(UNIT_DAYS)} steps/year)")

## Step 1: Build State Space
The domain's own `build_<domain>_states()`.

In [ ]:
print("=" * 60)
print(f"STEP 1: Building state space ({DOMAIN})")
print("=" * 60)

states = dom.build_states()
print(f"Total states: {len(states):,}")
print(f"\nState columns: {list(states.columns)}")
print("\nFirst 10 states:")
states.head(10)

## Step 2: Compute Probabilities

In [ ]:
print("=" * 60)
print(f"STEP 2: Computing probabilities ({DOMAIN})")
print("=" * 60)

probability_specs = getattr(dom.probabilities, "probability_specs", None)
if probability_specs:
    print("\nDeclared probability specs:")
    for spec in probability_specs:
        print(f"  {spec['name']:25s} depends on {str(spec['depends_on']):35s} active: {spec['method']}")

probs = dom.probabilities.compute_probabilities(states)
print(f"\nProbability arrays computed: {list(probs.keys())}")

## Step 3: Build Transition Matrix
The domain gives `(rows, cols, vals)`; the generic engine assembles and checks the matrix.

In [ ]:
print("=" * 60)
print(f"STEP 3: Building transition matrix ({DOMAIN})")
print("=" * 60)

rows, cols, vals = dom.transitions.compute_transitions(states, probs)
matrix = core.assemble_matrix(states, rows, cols, vals)
if STRICT:
    core.validate_row_stochasticity(matrix, states)
else:
    rs = row_sum_report(matrix)
    print(f"NOTE: STRICT is False for this domain: {rs['rows_not_summing_to_1']} rows do not sum to 1 "
          f"(min {rs['row_sum_min']:.4f}, max {rs['row_sum_max']:.4f}); kept as the domain defines them, not normalised.")
print(f"Nonzero entries: {matrix.nnz:,}")
print(f"Matrix shape: {matrix.shape}")
print(f"Sparsity: {matrix.nnz / (matrix.shape[0] * matrix.shape[1]) * 100:.2f}%")

## Step 4: Compute Reward Vector

In [ ]:
print("=" * 60)
print(f"STEP 4: Computing reward vector ({DOMAIN})")
print("=" * 60)

if HAS_REWARDS:
    rewards = dom.rewards.compute_rewards(states, probs)
    print(f"Reward components: {list(rewards.keys())}")
    print(f"Total reward range: [{rewards['total'].min():.2f}, {rewards['total'].max():.2f}]")
else:
    rewards = None
    print(f"Domain {DOMAIN!r} defines no rewards: the value steps (5, 6) and the reward exports are skipped.")

# Steps 1-4 in one call: build_model gives the identical model (same matrix checksum).
model = build_model(dom)
print(f"\nbuild_model(): {len(model.states):,} states, matrix checksum {matrix_checksum(model.P)} "
      f"({'identical to' if matrix_checksum(model.P) == matrix_checksum(matrix) else 'DIFFERENT FROM'} the stage-by-stage matrix)")

## Step 5: Finite-Horizon Bellman Evaluation
Full table (every state of the table). Reachable-state values follow in Step 6.

In [ ]:
print("=" * 60)
print("STEP 5: Finite-horizon Bellman evaluation")
print("=" * 60)

if HAS_REWARDS:
    V_finite = core.evaluate_finite_horizon(matrix, rewards["total"], gamma, NUM_YEARS, UNIT_DAYS)
else:
    print("skipped: no rewards")

## Step 6: Infinite-Horizon Exact Solve
**Full-table versus reachable-state results are different quantities.** The table can contain states no animal reaches under the default settings
(cow, month: 2,970 states of which 1,210 reachable from the replacement state). Full-table means average over all of them. The reachable set is
closed under the transitions, so its values equal the full-table values on those states, and solving only it is cheaper (`evaluate(..., start=...)`).

In [ ]:
print("=" * 60)
print("STEP 6: Infinite-horizon exact solve")
print("=" * 60)

if HAS_REWARDS:
    V_infinite = core.evaluate_infinite_horizon(matrix, rewards["total"], gamma)
    print(f"mean V = {V_infinite.mean():.2f}  min = {V_infinite.min():.2f}  max = {V_infinite.max():.2f}   (scope: full table, {len(states):,} states)")

    start = "replacement" if dom.hook("replacement_state_id") else "initial"
    reach = evaluate(model, dom, GAMMA_ANNUAL, NUM_YEARS, start=start)
    full = evaluate(model, dom, GAMMA_ANNUAL, NUM_YEARS)
    print(pd.DataFrame({"scope": [full["scope"], reach["scope"]], "states": [full["n_states"], reach["n_states"]],
                        f"mean V ({NUM_YEARS} y)": [full["V_finite"].mean(), reach["V_finite"].mean()],
                        "mean V (infinite)": [full["V_infinite"].mean(), reach["V_infinite"].mean()]}).to_string(index=False))
    same = np.allclose(full["V_infinite"][reach["state_ids"]], reach["V_infinite"], rtol=1e-9, atol=1e-7)
    print(f"values on the reachable states: full-table solve == reachable-subsystem solve: {same}")

    top_states = states.copy()
    top_states["V_infinite"] = V_infinite
    display(top_states.nlargest(10, "V_infinite"))
else:
    print("skipped: no rewards")

## Step 7: State Distribution Evolution (3 Methods)

In [ ]:
print("=" * 60)
print("STEP 7: State distribution evolution")
print("=" * 60)

pi0 = initial_distribution(model, dom)          # the domain's INITIAL_STATE hook, else state_id 0
print(f"Initial distribution: 1.0 mass at state {int(np.argmax(pi0))}")

num_steps = core.steps_per_year(UNIT_DAYS)
print(f"\nEvolving {num_steps} steps ahead (1 year)...")

def _line(label, pi):
    extra = f"  expected reward = {pi @ rewards['total']:.2f}" if HAS_REWARDS else ""
    print(f"{label} sum = {pi.sum():.6f}{extra}")

pi_recursive = evolve_distribution(model, pi0, dom, num_steps)
_line("Method 1 (recursive):     ", pi_recursive)
if len(states) <= 3000:
    _line("Method 2 (matrix power):  ", evolve_distribution(model, pi0, dom, num_steps, method="matrix_power"))
    _line("Method 3 (eigen):         ", evolve_distribution(model, pi0, dom, num_steps, method="eigen"))
else:
    print(f"Skipping Methods 2 & 3: {len(states):,} states is too many for dense matrix operations.")

## Step 7b: Replacement-Aware Distribution Evolution
Splits the evolving distribution into "never replaced since t=0" vs "replaced at least once" using the domain's own `replacement_state_id` hook. Skipped when the domain has none.

In [ ]:
print("=" * 60)
print("STEP 7b: Replacement-aware distribution evolution")
print("=" * 60)

orig_trajectory = tagged_trajectory = None
if dom.hook("replacement_state_id") is None:
    print("skipped: this domain has no replacement_state_id hook")
else:
    replacement_id = dom.hook("replacement_state_id")(states)
    print(f"Replacement state id: {replacement_id} -> {states.iloc[replacement_id][[c for c in states.columns if c != 'state_id']].to_dict()}")
    orig_trajectory, tagged_trajectory = replacement_split(model, pi0, dom, num_steps)
    if STRICT:
        for step, (po, pt) in enumerate(zip(orig_trajectory, tagged_trajectory)):
            assert np.isclose((po + pt).sum(), 1.0, atol=1e-9), f"Mass not conserved at step {step}"
    replaced_fraction = [pt.sum() for pt in tagged_trajectory]
    print(f"\nFraction of the herd replaced at least once, by {TIME_UNIT} (1 MRP step = 1 {TIME_UNIT}):")
    for step in range(0, num_steps + 1, max(1, num_steps // 10)):
        print(f"  {TIME_UNIT.capitalize()} {step:3d}: {replaced_fraction[step]*100:5.1f}% replaced")

## Step 8: Export Results
`export_model` writes the states, the transition matrix (long table and sparse `.npz`) and the reward vector. The state-variable column names come from the domain (`EXPORT_COLUMN_NAMES`), not from the engine.

In [ ]:
print("=" * 60)
print("STEP 8: Exporting results")
print("=" * 60)

files = export_model(model, dom, OUT_DIR, parquet=False)       # parquet needs the optional pyarrow
for f in files:
    print("Saved:", os.path.basename(f))
print("\n" + "=" * 60)
print(f"DONE! ({DOMAIN} domain)")
print("=" * 60)

## Step 9: Stochastic Trajectory Simulation
Domain-agnostic: sample individual animals through the same matrix. `seed` makes it reproducible.

In [ ]:
print("=" * 60)
print("STEP 9a: Single Stochastic Trajectory")
print("=" * 60)

n_entities = 100
initial_state_id = int(np.argmax(pi0))
initial_state_ids = np.full(n_entities, initial_state_id)
n_steps = core.steps_per_year(UNIT_DAYS)  # 1 year

history = simulate(model, initial_state_ids, n_steps, seed=42)
print(f"History shape: {history.shape}  (steps+1, entities)")
print("\nFirst 5 entities' state IDs over time:")
print(history[:, :5])

In [ ]:
print("=" * 60)
print("STEP 9b: Monte Carlo Simulation")
print("=" * 60)

n_sims = 50
trajectories = monte_carlo(model, initial_state_ids, n_steps, n_sims=n_sims, seed=42)
print(f"Returned {len(trajectories)} trajectories")
final_in_initial = [int(np.sum(traj[-1] == initial_state_id)) for traj in trajectories]
print("\nFinal step: entities still in initial state")
print(f"  Mean: {np.mean(final_in_initial):.1f}   Std: {np.std(final_in_initial):.1f}   Min: {np.min(final_in_initial)}   Max: {np.max(final_in_initial)}")

In [ ]:
print("=" * 60)
print("STEP 9c: Monte Carlo Grouped Summary")
print("=" * 60)

if dom.hook("group_labels") is None:
    print("skipped: this domain has no group_labels hook")
else:
    print(f"Grouping: {GROUP_NAME}   (counts per label at the final step, across {n_sims} sims)")
    print(group_summary(model, dom, trajectories).round(1).to_string(index=False))

## Step 10: Interactive Plotly Visualization
Optional (needs plotly). The charts need the hooks of Steps 7b and 9c and are skipped without them.

In [ ]:
print("=" * 60)
print("STEP 10: Interactive Plotly visualization")
print("=" * 60)

try:
    from cowmrp.viz.distribution import replacement_split_figure, distribution_by_group
except ImportError as e:
    print("WARNING: plotly not installed (pip install cowmrp[plot]) -- skipping this cell.", e)
else:
    if orig_trajectory is None:
        print("skipped: the charts need the replacement_state_id hook (Step 7b)")
    else:
        if dom.hook("group_labels") is not None:
            distribution_by_group(model, dom, orig_trajectory, tagged_trajectory).show()
        replacement_split_figure(dom, orig_trajectory, tagged_trajectory).show()

## Write your own domain
The engine needs nothing from the animal. Provide a package with the files listed at the top (`states.py` with `build_<name>_states`,
`probabilities`, `transitions.py`, optional `rewards`, `parameters.py`, optional `notebook_hooks.py`), then:

```python
dom = load_domain("my_package.my_domain")
print(validate_domain(dom))      # lists anything missing
model = build_model(dom)
```
The full contract is in `cowmrp.domain`.